# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Stage 1 — Tier 3 metric: stereotype flattening $\Delta H_c$

Reads the scored parquets and computes the stereotype-flattening metric
specified in §2.2 eq. (8) of the report:
$$
\Delta H_c = \frac{1}{n_c} \sum_{i:Y_i=c}\!\left[
  \hat H_{\text{MM}}\!\left[\hat P_{\text{ACS}}(Y \mid \mathrm{cell}(X_{-Y,i}))\right]
  - H\!\left[P_{\text{LLM}}(Y \mid X_{-Y,i})\right]
\right]
$$
where $\hat H_{\text{MM}}(P) = \hat H(P) + (K-1)/(2m)$ is the
Miller–Madow bias-corrected entropy estimator with $K = n_Y$ classes and
$m$ samples in the cell.

**$\Delta H_c > 0$ means the LLM is sharper than ACS** (under-represents
within-group heterogeneity for class $c$) — the flattening signal that
Wang et al. (2024) flag as the key failure mode of LLM-as-survey-respondent.

**Cell definition.** Per §2.2, default cells are race × age-bucket. When
$Y$ overlaps with a default axis ($Y \in \{$RACE_ETHNICITY, AGE_BINS$\}$),
substitute education × employment-status. We use the existing `AGE_BINS`
column directly rather than constructing 5-year buckets manually
(raw AGE was not preserved in the scored parquet schema). The 10-year
sensitivity check from the report is therefore deferred to a post-hoc
ablation if needed.

**ACS-side estimation.** Per-cell $\hat P_{\text{ACS}}(Y \mid \text{cell})$
is estimated from the eval set itself using **Hájek weights**, which
recover the population-conditional from the stratified design:
$$\hat P_{\text{ACS}}(Y = c \mid \text{cell}) =
   \frac{\sum_{i \in \text{cell}, Y_i = c} w_i}
        {\sum_{i \in \text{cell}} w_i}.$$
Cells with $\sum w_i < 50$ are excluded from the per-row average. Rows
whose cell falls below threshold simply don't contribute to $\Delta H_c$.

**LLM-side entropy.** Per-row entropy of the LLM's debiased conditional
$P_{\text{LLM}}(Y \mid X_{-Y,i})$ — the row's full predicted distribution,
no further binning. We deliberately don't coarsen the LLM side, so the
comparison is "ACS-cell heterogeneity vs LLM per-row heterogeneity," which
is conservative for detecting flattening (any per-row heterogeneity the
LLM produces is credited).

**Section overview**

| §  | Output                                                       |
|----|--------------------------------------------------------------|
| 1  | Setup: load parquets                                         |
| 2  | Cell construction per target                                 |
| 3  | $\Delta H_c$ computation (Miller–Madow + per-row averaging) |
| 4  | Per-target $\Delta H_c$ tables (raw + debiased side-by-side)|
| 5  | Cross-target summary                                         |
| 6  | Headline figure: per-class $\Delta H_c$ with cell coverage  |


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Model under analysis ---
# MODEL_NAME = "mistralai/Ministral-3-14B-Instruct-2512"
#MODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507"
#MODEL_NAME = "Qwen3-4B-Instruct-2507-FT_cumulative_bf16_05_06"
#MODEL_NAME = "Qwen3-4B-Instruct-2507-FT_cumulative_bf16_05_07"
#MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507_05_05"

#MODEL_NAME = "Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_balanced_05_11"
# MODEL_NAME = "Qwen2.5-14B-Instruct_baseline_test_balanced_05_11"
#MODEL_NAME = "Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_raw_05_12_lauren"
# MODEL_NAME = "Qwen2.5-14B-Instruct-FT_balanced_canonical_step07_bf16_test_raw_05_12"
MODEL_NAME = "Qwen2.5-14B-Instruct_baseline_test_raw_05_12"
# --- Display ---
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

# --- Config ---
CELL_MIN_WEIGHT = 50.0    # minimum sum(hajek_weight) for a cell to count
GLOBAL_SEED = 42

# --- I/O ---
SCORED_DIR  = Path(
    f"/content/drive/MyDrive/NLP_project/data/"
    f"stage_1_output_{MODEL_NAME}/"
)
METRICS_DIR = Path(
    f"/content/drive/MyDrive/NLP_project/data/"
    f"stage_1_output_{MODEL_NAME}/phase1_metrics/"
)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "SEX",
    "AGE_BINS",
    "RACE_ETHNICITY",
    "EDUCATION_LEVEL",
    "INCTOT_BINS",
    "HEALTH_INSURANCE",
    "ENGLISH_STATUS",
]


In [ ]:
def detect_prob_kinds(scored: pd.DataFrame, classes: list) -> list:
    has_raw = all(f"prob_raw_{c}" in scored.columns for c in classes)
    has_deb = all(f"prob_deb_{c}" in scored.columns for c in classes)
    if has_raw and has_deb:
        return ["raw", "deb"]
    return ["raw"]


def load_target(target: str) -> dict:
    pq_path  = SCORED_DIR / f"phase1_scored_{target}.parquet"
    meta_path = SCORED_DIR / f"phase1_scored_{target}.meta.json"
    if not pq_path.exists():
        raise FileNotFoundError(f"Missing scored parquet for {target}: {pq_path}")
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing metadata for {target}: {meta_path}")
    scored = pd.read_parquet(pq_path)
    with open(meta_path) as f:
        meta = json.load(f)
    classes = meta["class_labels"]
    prob_kinds = detect_prob_kinds(scored, classes)
    needs_legacy = (prob_kinds == ["raw"]
                    and not all(f"prob_raw_{c}" in scored.columns for c in classes))
    if needs_legacy:
        for c in classes:
            scored[f"prob_raw_{c}"] = scored[f"prob_{c}"]
    return {
        "scored":        scored,
        "meta":          meta,
        "classes":       classes,
        "prob_kinds":    prob_kinds,
        "applied_pride": meta.get("apply_pride", False),
    }


def prob_cols(classes: list, kind: str = "deb") -> list:
    return [f"prob_{kind}_{c}" for c in classes]


runs = {}
for t in TARGETS:
    try:
        runs[t] = load_target(t)
        n = len(runs[t]["scored"])
        k = len(runs[t]["classes"])
        kinds = "+".join(runs[t]["prob_kinds"])
        pride = "PriDe" if runs[t]["applied_pride"] else "raw-only"
        print(f"  {t:<20} loaded: {n:,} rows, {k} classes, "
              f"kinds=[{kinds}] ({pride})")
    except FileNotFoundError as e:
        warnings.warn(f"Skipping {t}: {e}")

print(f"\nLoaded {len(runs)} / {len(TARGETS)} targets.")


## 2. Cell construction

For each target $Y$, we choose a pair of demographic axes and form
cells as their cross-product.

| target $Y$        | cell axes                                  |
|-------------------|--------------------------------------------|
| RACE_ETHNICITY    | EDUCATION_LEVEL × EMPLOYMENT_STATUS        |
| AGE_BINS          | EDUCATION_LEVEL × EMPLOYMENT_STATUS        |
| (any other)       | RACE_ETHNICITY × AGE_BINS                  |

The cell label for a row is just the concatenated string of the two
axis values, e.g. `"White|25-34"`. We use `AGE_BINS` directly — no
manual age binning — and any row with NaN on either axis is dropped
from the $\Delta H_c$ computation (those rows contribute neither to
the ACS-side cell estimate nor to the per-row average).


In [ ]:
def cell_axes_for_target(target: str) -> tuple:
    """Return the two column names used to form cells for this target.

    Default: race x age-bucket. Substitute when target overlaps either axis.
    """
    if target in ("RACE_ETHNICITY", "AGE_BINS"):
        return ("EDUCATION_LEVEL", "EMPLOYMENT_STATUS")
    return ("RACE_ETHNICITY", "AGE_BINS")


def make_cell_labels(scored: pd.DataFrame, target: str) -> pd.Series:
    """Assign a cell label per row using the target's cell axes.

    Returns a string Series aligned with `scored`. NaN axes propagate to
    NaN cell label so downstream code can mask them out.
    """
    ax1, ax2 = cell_axes_for_target(target)
    if ax1 not in scored.columns or ax2 not in scored.columns:
        raise KeyError(
            f"Cell axes {ax1!r} or {ax2!r} not found in scored df for {target}. "
            f"Available columns: {list(scored.columns)[:30]}..."
        )
    a = scored[ax1].astype("string")
    b = scored[ax2].astype("string")
    cells = a + "|" + b
    cells[a.isna() | b.isna()] = pd.NA
    return cells


# Sanity: print the cell axis pairs and a few sample cell labels per target.
for t in runs:
    ax1, ax2 = cell_axes_for_target(t)
    cells = make_cell_labels(runs[t]["scored"], t)
    n_cells = cells.dropna().nunique()
    print(f"  {t:<20} cells = {ax1} x {ax2}  -> {n_cells} unique non-null cells")


## 3. $\Delta H_c$ computation

Three pieces:

1. **Per-cell ACS-conditional estimate** $\hat P_{\text{ACS}}(Y \mid \text{cell})$
   via Hájek-weighted within-cell counts.
2. **Miller–Madow correction** $\hat H_{\text{MM}}(P) = \hat H(P) + (K-1)/(2m)$
   where $K = n_Y$ and $m$ = sum of Hájek weights in the cell.
3. **Per-row LLM entropy** $H(P_{\text{LLM}}(Y \mid X_{-Y,i}))$ in nats.

Then for each true class $c$, average
$\hat H_{\text{MM}}(\hat P_{\text{ACS}}(\cdot \mid \text{cell}_i)) - H(P_{\text{LLM}, i})$
over rows in class $c$ whose cell met the $\sum w_i \geq 50$ threshold.


In [ ]:
def shannon_entropy(P: np.ndarray, axis: int = -1) -> np.ndarray:
    """Shannon entropy in nats. Handles zero probabilities via 0 * log 0 = 0."""
    with np.errstate(divide="ignore", invalid="ignore"):
        log_P = np.where(P > 0, np.log(P), 0.0)
        return -np.sum(P * log_P, axis=axis)


def miller_madow_correction(K: int, m: float) -> float:
    """(K-1) / (2m). Guards against m=0 by returning 0 (cell will be excluded)."""
    if m <= 0:
        return 0.0
    return (K - 1) / (2.0 * m)


def acs_conditional_per_cell(scored: pd.DataFrame, target_col: str,
                              classes: list, cell_col: str = "_cell"
                              ) -> pd.DataFrame:
    """Per-cell P_ACS(Y | cell) and total Hajek weight per cell.

    Returns a DataFrame indexed by cell label with one column per class
    plus 'm_cell' (sum of Hajek weights in cell).
    """
    if "hajek_weight" not in scored.columns:
        raise KeyError("hajek_weight column missing; cannot estimate P_ACS.")
    df = scored[[cell_col, target_col, "hajek_weight"]].dropna(subset=[cell_col])
    # Sum of weights per (cell, class)
    sums = (df.groupby([cell_col, target_col], observed=True)["hajek_weight"]
              .sum()
              .unstack(target_col, fill_value=0.0))
    # Reindex columns to canonical class order; missing classes become 0.
    sums = sums.reindex(columns=classes, fill_value=0.0)
    m_cell = sums.sum(axis=1)
    P_acs = sums.div(m_cell.replace(0, np.nan), axis=0)
    P_acs["m_cell"] = m_cell
    return P_acs


def compute_delta_h(scored: pd.DataFrame, target: str, classes: list,
                     kind: str, cell_min: float = CELL_MIN_WEIGHT) -> dict:
    """Compute per-class Delta H_c plus diagnostic stats.

    Returns dict with:
      per_class_delta_h : (n_classes,) array of mean Delta H_c values
      per_class_n_used  : (n_classes,) rows in class with above-threshold cell
      per_class_n_total : (n_classes,) rows in class total
      per_class_h_acs   : (n_classes,) mean H_MM (ACS) for that class
      per_class_h_llm   : (n_classes,) mean H (LLM) for that class
      n_cells_total     : int, unique cells before threshold
      n_cells_kept      : int, unique cells with m_cell >= cell_min
      coverage_overall  : fraction of rows with above-threshold cell
    """
    n_Y = len(classes)
    df = scored.copy()
    df["_cell"] = make_cell_labels(df, target)

    # ACS-side per-cell distribution
    P_acs_per_cell = acs_conditional_per_cell(df, target, classes, "_cell")
    n_cells_total = int(P_acs_per_cell.shape[0])

    # Threshold cells
    keep_mask = P_acs_per_cell["m_cell"] >= cell_min
    P_acs_kept = P_acs_per_cell[keep_mask].copy()
    n_cells_kept = int(P_acs_kept.shape[0])

    # H_MM per cell
    P_only = P_acs_kept[classes].to_numpy()
    H_acs_raw = shannon_entropy(P_only, axis=1)
    mm_correction = np.array(
        [miller_madow_correction(n_Y, m) for m in P_acs_kept["m_cell"].to_numpy()]
    )
    H_acs_mm = H_acs_raw + mm_correction
    P_acs_kept["H_acs_mm"] = H_acs_mm

    # Map per-row to its cell's H_acs_mm (NaN if cell below threshold or NaN)
    H_acs_per_row = df["_cell"].map(P_acs_kept["H_acs_mm"]).to_numpy()

    # LLM-side per-row entropy
    P_llm = df[prob_cols(classes, kind)].to_numpy()
    H_llm_per_row = shannon_entropy(P_llm, axis=1)

    # True-class index per row
    true_col = df[f"true_{target}"].astype("string")
    cls_to_idx = {c: i for i, c in enumerate(classes)}
    y_idx = true_col.map(cls_to_idx).to_numpy()

    # Per-class average
    per_class_delta_h = np.full(n_Y, np.nan)
    per_class_n_used  = np.zeros(n_Y, dtype=np.int64)
    per_class_n_total = np.zeros(n_Y, dtype=np.int64)
    per_class_h_acs   = np.full(n_Y, np.nan)
    per_class_h_llm   = np.full(n_Y, np.nan)

    valid = ~np.isnan(H_acs_per_row) & ~np.isnan(y_idx.astype(float))
    for c_idx in range(n_Y):
        in_class_total = (y_idx == c_idx)
        in_class_used  = in_class_total & valid
        per_class_n_total[c_idx] = int(in_class_total.sum())
        per_class_n_used[c_idx]  = int(in_class_used.sum())
        if per_class_n_used[c_idx] > 0:
            h_acs = H_acs_per_row[in_class_used]
            h_llm = H_llm_per_row[in_class_used]
            per_class_h_acs[c_idx] = float(np.mean(h_acs))
            per_class_h_llm[c_idx] = float(np.mean(h_llm))
            per_class_delta_h[c_idx] = float(np.mean(h_acs - h_llm))

    coverage = float(valid.mean())

    return {
        "per_class_delta_h": per_class_delta_h,
        "per_class_n_used":  per_class_n_used,
        "per_class_n_total": per_class_n_total,
        "per_class_h_acs":   per_class_h_acs,
        "per_class_h_llm":   per_class_h_llm,
        "n_cells_total":     n_cells_total,
        "n_cells_kept":      n_cells_kept,
        "coverage_overall":  coverage,
    }


# Compute for every (target, kind)
delta_h_data = {}
for t, run in runs.items():
    for kind in run["prob_kinds"]:
        try:
            delta_h_data[(t, kind)] = compute_delta_h(
                run["scored"], t, run["classes"], kind, cell_min=CELL_MIN_WEIGHT,
            )
            d = delta_h_data[(t, kind)]
            print(f"  {t:<20} {kind:<5} cells: {d['n_cells_kept']}/{d['n_cells_total']} "
                  f"kept   coverage: {d['coverage_overall']:.1%}")
        except (KeyError, ValueError) as e:
            warnings.warn(f"Skipping {t} ({kind}): {e}")

print(f"\nComputed Delta H_c for {len(delta_h_data)} (target, kind) pairs.")


## 4. Per-target $\Delta H_c$ tables

For each (target, kind), one row per class with:

- **n_used** / **n_total**: rows in class that landed in a kept cell
- **H_acs**: mean ACS-side entropy (with Miller–Madow correction)
- **H_llm**: mean LLM-side entropy (per-row, no correction)
- **delta_h**: $\hat H_{\text{MM}}^{\text{ACS}} - H^{\text{LLM}}$ (positive = LLM sharper than ACS)


In [ ]:
flattening_tables = {}

for (t, kind), d in delta_h_data.items():
    classes = runs[t]["classes"]
    rows = []
    for c_idx, c in enumerate(classes):
        rows.append({
            "class":    c,
            "n_used":   int(d["per_class_n_used"][c_idx]),
            "n_total":  int(d["per_class_n_total"][c_idx]),
            "H_acs":    d["per_class_h_acs"][c_idx],
            "H_llm":    d["per_class_h_llm"][c_idx],
            "delta_h":  d["per_class_delta_h"][c_idx],
        })
    tbl = pd.DataFrame(rows).set_index("class")
    flattening_tables[(t, kind)] = tbl
    out_path = METRICS_DIR / f"flattening_{t}_{kind}.csv"
    tbl.to_csv(out_path)

# Display debiased tables
print("=" * 80)
print("PER-TARGET FLATTENING TABLES (debiased)")
print("=" * 80)
print("delta_h > 0 means LLM is sharper than ACS (under-represents heterogeneity)")
print()
for t in runs:
    kind = "deb" if (t, "deb") in flattening_tables else "raw"
    if (t, kind) not in flattening_tables:
        continue
    print(f"\n[{t}]  kind={kind}  cells_kept={delta_h_data[(t, kind)]['n_cells_kept']}")
    print(flattening_tables[(t, kind)].to_string(float_format=lambda x: f"{x:.3f}"))


## 5. Cross-target summary

One row per target with population-style summaries:

- **mean_delta_h**: unweighted mean of per-class $\Delta H_c$
- **max_delta_h**: most-flattened class (worst case)
- **min_delta_h**: least-flattened class (could be negative if LLM is *more* heterogeneous than ACS)
- **flat_classes**: number of classes with $\Delta H_c > 0$ (LLM sharper)
- **n_classes_evaluated**: classes with usable data after threshold

This is the table for the report's flattening section.


In [ ]:
summary_rows = []
for t in runs:
    kind = "deb" if (t, "deb") in delta_h_data else "raw"
    if (t, kind) not in delta_h_data:
        continue
    d = delta_h_data[(t, kind)]
    dh = d["per_class_delta_h"]
    valid = ~np.isnan(dh)
    if not valid.any():
        continue
    summary_rows.append({
        "target":              t,
        "n_classes":           len(runs[t]["classes"]),
        "kind":                kind,
        "n_classes_eval":      int(valid.sum()),
        "cells_kept":          d["n_cells_kept"],
        "cells_total":         d["n_cells_total"],
        "coverage":            d["coverage_overall"],
        "mean_delta_h":        float(np.nanmean(dh)),
        "max_delta_h":         float(np.nanmax(dh)),
        "min_delta_h":         float(np.nanmin(dh)),
        "flat_classes":        int((dh > 0).sum()),
    })

flattening_summary = pd.DataFrame(summary_rows).set_index("target")
print("CROSS-TARGET FLATTENING SUMMARY  (debiased)")
print("=" * 80)
print("mean_delta_h > 0 => LLM produces sharper conditionals than ACS")
print("flat_classes / n_classes_eval = fraction of classes where LLM is sharper")
print()
print(flattening_summary.to_string(float_format=lambda x: f"{x:.3f}"))

flattening_summary_path = METRICS_DIR / "flattening_cross_target_summary.csv"
flattening_summary.to_csv(flattening_summary_path)
print(f"\nSaved -> {flattening_summary_path}")


## 6. Headline figure: per-class $\Delta H_c$

One panel per target, horizontal bars showing $\Delta H_c$ per class.
Bars to the right of zero = LLM sharper than ACS (the flattening signal).
Bars to the left = LLM more heterogeneous than ACS (rare). Class-name
fontsize and ordering follow the convention from the Tier 1 Brier figure.

Bar color encodes whether the cell coverage for that class was sufficient:
darker bars = $n_{\text{used}} \geq 100$ rows; lighter = sparse class.


In [ ]:
n_targets = len(runs)
ncols = 2
nrows = (n_targets + ncols - 1) // ncols

# Determine common x-limits so panels are comparable
all_dh = []
for (t, kind), d in delta_h_data.items():
    if kind == ("deb" if (t, "deb") in delta_h_data else "raw"):
        all_dh.extend(d["per_class_delta_h"][~np.isnan(d["per_class_delta_h"])].tolist())
xlim_max = max(0.5, np.nanmax(np.abs(all_dh)) * 1.1) if all_dh else 1.0

fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols,
    figsize=(7 * ncols, 2.5 + 1.0 * max(len(r["classes"]) for r in runs.values())),
    squeeze=False,
)

for i, t in enumerate(runs):
    ax = axes[i // ncols, i % ncols]
    kind = "deb" if (t, "deb") in delta_h_data else "raw"
    if (t, kind) not in delta_h_data:
        ax.set_visible(False); continue
    classes = runs[t]["classes"]
    d = delta_h_data[(t, kind)]
    dh = d["per_class_delta_h"]
    n_used = d["per_class_n_used"]

    # Sort by delta_h ascending (least flattened at top)
    order = np.argsort(np.where(np.isnan(dh), -np.inf, dh))
    classes_sorted = [classes[i] for i in order]
    dh_sorted = dh[order]
    n_used_sorted = n_used[order]

    # Color by n_used
    colors = ["#3a6f9b" if n >= 100 else "#9bb8d3" for n in n_used_sorted]

    y_pos = np.arange(len(classes_sorted))
    ax.barh(y_pos, dh_sorted, color=colors, edgecolor="white", linewidth=0.5)

    # Per-class n annotations on the right edge
    for yi, (val, n_u) in enumerate(zip(dh_sorted, n_used_sorted)):
        if np.isnan(val):
            ax.text(0.01, yi, f"  n={n_u}  (no cell)", va="center", fontsize=7, color="#888")
        else:
            ax.text(val + 0.02 if val >= 0 else val - 0.02, yi, f"n={n_u}",
                    va="center",
                    ha="left" if val >= 0 else "right",
                    fontsize=7, color="#444")

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(classes_sorted, fontsize=9)
    ax.set_xlabel(r"$\Delta H_c$ (nats; positive = LLM sharper)")
    ax.set_title(f"{t}  ({kind}, n_Y={len(classes)}, "
                 f"cells={d['n_cells_kept']}/{d['n_cells_total']})",
                 fontsize=11)
    ax.set_xlim(-xlim_max, xlim_max)
    ax.grid(axis="x", linestyle=":", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)

# Hide unused subplots
for j in range(n_targets, nrows * ncols):
    axes[j // ncols, j % ncols].set_visible(False)

fig.suptitle(
    r"Stereotype flattening $\Delta H_c = H_{\mathrm{MM}}^{\mathrm{ACS}} - H^{\mathrm{LLM}}$ "
    r"(debiased)",
    fontsize=13, y=1.00,
)
plt.tight_layout()
plt.savefig(METRICS_DIR / "fig4_flattening.png", dpi=180, bbox_inches="tight")
plt.savefig(METRICS_DIR / "fig4_flattening.pdf", bbox_inches="tight")
plt.show()
